In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.optimize import minimize

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score


tqdm.pandas()

RANDOM_STATE = 42
FORCE_TRAIN = False

TRAIN_PATH = "/content/drive/MyDrive/CS114/TeamModel/data/train_clean_tokenizedEmoji.csv"
VAL_PATH = "/content/drive/MyDrive/CS114/TeamModel/data/val_clean_tokenizedEmoji.csv"
TEST_PATH = "/content/drive/MyDrive/CS114/TeamModel/data/test_clean_tokenizedEmoji.csv"

MODEL_DIR = "/content/drive/MyDrive/CS114/TeamModel/models"
REPORT_DIR = "/content/drive/MyDrive/CS114/TeamModel/reports"
MODEL_PATH = os.path.join(MODEL_DIR, "lr_lsa_TFIDF.joblib")
BEST_WEIGHTS_PATH = os.path.join(MODEL_DIR, "lr_lsa_TFIDF_best_threshold_weights.json")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)



In [3]:
print(">> [1/5] Đang nạp tập dữ liệu...")
train_raw = pd.read_csv(TRAIN_PATH).dropna(subset=["text", "status"])
val_raw = pd.read_csv(VAL_PATH).dropna(subset=["text", "status"])
test_raw = pd.read_csv(TEST_PATH).dropna(subset=["text", "status"])

train_raw.drop_duplicates(subset=["text"], inplace=True)


def prepare_text_dataframe(dataframe, name="Dataset"):

    df_out = dataframe.copy()
    print(f"\nChuẩn hóa text: {name} ({len(df_out)} mẫu)")

    df_out["cleaned_text"] = df_out["text"].astype(str).str.lower().str.strip()
    df_out["cleaned_text"] = df_out["cleaned_text"].progress_apply(
        lambda x: re.sub(r"(.)\1{4,}", r"\1\1", x)
    )
    return df_out


train_enriched = prepare_text_dataframe(train_raw, "TRAIN SET")
val_enriched = prepare_text_dataframe(val_raw, "VALIDATION SET")
test_enriched = prepare_text_dataframe(test_raw, "TEST SET")

feature_cols = ["cleaned_text"]

X_train, y_train = train_enriched[feature_cols], train_enriched["status"].str.strip().str.lower()
X_val, y_val = val_enriched[feature_cols], val_enriched["status"].str.strip().str.lower()
X_test, y_test = test_enriched[feature_cols], test_enriched["status"].str.strip().str.lower()


>> [1/5] Đang nạp tập dữ liệu...

Chuẩn hóa text: TRAIN SET (34585 mẫu)


  0%|          | 0/34585 [00:00<?, ?it/s]


Chuẩn hóa text: VALIDATION SET (4943 mẫu)


  0%|          | 0/4943 [00:00<?, ?it/s]


Chuẩn hóa text: TEST SET (9887 mẫu)


  0%|          | 0/9887 [00:00<?, ?it/s]

In [4]:
# TF-IDF + LSA + Logistic Regression

print(">> [2/5] Biên dịch TF-IDF + LSA + Logistic Regression...")

lsa_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        max_features=30000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        dtype=np.float32
    )),
    ("svd", TruncatedSVD(
        n_components=300,
        algorithm="randomized",
        random_state=RANDOM_STATE
    )),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("lsa", lsa_pipeline, "cleaned_text")
], verbose=True)

pipeline_ensemble = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        C=1.0,
        solver="saga",
        penalty="l2",
        max_iter=5000,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
], verbose=True)



>> [2/5] Biên dịch TF-IDF + LSA + Logistic Regression...


In [5]:
if os.path.exists(MODEL_PATH) and not FORCE_TRAIN:
    print(f">> [3/5] Nạp mô hình Logistic Regression từ tệp tin: {MODEL_PATH}")
    pipeline_ensemble = joblib.load(MODEL_PATH)
else:
    print(">> [3/5] Huấn luyện TF-IDF + LSA + Logistic Regression...")
    pipeline_ensemble.fit(X_train, y_train)
    joblib.dump(pipeline_ensemble, MODEL_PATH)
    print(f"Đã lưu mô hình thành công tại: {MODEL_PATH}")



>> [3/5] Nạp mô hình Logistic Regression từ tệp tin: /content/drive/MyDrive/CS114/TeamModel/models/lr_lsa_TFIDF.joblib


In [6]:
print("\n>> [4/5] Tối ưu hóa trọng số xác suất phân loại trên validation set...")
proba_train = pipeline_ensemble.predict_proba(X_train)
proba_val = pipeline_ensemble.predict_proba(X_val)
proba_test = pipeline_ensemble.predict_proba(X_test)

class_names = pipeline_ensemble.classes_
class_mapping = {label: idx for idx, label in enumerate(class_names)}
y_val_numeric = np.array([class_mapping[l] for l in y_val])


def macro_f1_loss(weights, probabilities, y_true):
    scaled_proba = probabilities * weights
    preds = np.argmax(scaled_proba, axis=1)
    return -f1_score(y_true, preds, average="macro")


init_weights = np.ones(len(class_names), dtype=np.float32)
bounds = [(0.1, 10.0) for _ in range(len(class_names))]

opt_result = minimize(
    macro_f1_loss,
    init_weights,
    args=(proba_val, y_val_numeric),
    method="Nelder-Mead",
    bounds=bounds,
    options={"maxiter": 500}
)

best_weights = opt_result.x
print(f"Trọng số hiệu chỉnh tìm được: {best_weights}")

with open(BEST_WEIGHTS_PATH, "w", encoding="utf-8") as f:
    json.dump({cls: float(w) for cls, w in zip(class_names, best_weights)}, f, indent=4, ensure_ascii=False)
print(f"Đã lưu trọng số threshold/probability calibration tại: {BEST_WEIGHTS_PATH}")


def predict_with_weights(probabilities, weights, classes):
    scaled_proba = probabilities * weights
    pred_indices = np.argmax(scaled_proba, axis=1)
    return np.array([classes[idx] for idx in pred_indices])


preds_train_final = predict_with_weights(proba_train, best_weights, class_names)
preds_val_final = predict_with_weights(proba_val, best_weights, class_names)
preds_test_final = predict_with_weights(proba_test, best_weights, class_names)



>> [4/5] Tối ưu hóa trọng số xác suất phân loại trên validation set...
Trọng số hiệu chỉnh tìm được: [0.47919629 1.30932634 1.26471762 1.07874911]
Đã lưu trọng số threshold/probability calibration tại: /content/drive/MyDrive/CS114/TeamModel/models/lr_lsa_TFIDF_best_threshold_weights.json


In [7]:

f1_train = f1_score(y_train, preds_train_final, average='macro')
f1_val = f1_score(y_val, preds_val_final, average='macro')
f1_test = f1_score(y_test, preds_test_final, average='macro')
gap = f1_train - f1_val

print("\n" + "="*50)
print("     KẾT QUẢ ĐÁNH GIÁ")
print("="*50)
print(f"Macro F1 - Tập Huấn luyện (Train) : {f1_train:.4f}")
print(f"Macro F1 - Tập Kiểm định  (Val)   : {f1_val:.4f}")
print(f"Macro F1 - Tập Kiểm thử   (Test)  : {f1_test:.4f}")
print(f"Khoảng sai lệch tổng quát hóa     : {gap:.4f}")
print("="*50)

print("\n>> [5/5] Lưu báo cáo chi tiết...")
report_path = os.path.join(REPORT_DIR, "lr_lsa_TFIDF_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("==================================================\n")
    f.write("  BÁO CÁO KẾT QUẢ\n")
    f.write("==================================================\n\n")

    splits = [("TRAIN SET", y_train, preds_train_final),
              ("VALIDATION SET", y_val, preds_val_final),
              ("TEST SET", y_test, preds_test_final)]

    for name, y_true, y_pred in splits:
        f.write(f"--- ĐÁNH GIÁ {name} ---\n")
        f.write(classification_report(y_true, y_pred, digits=4))
        f.write("\n" + "="*50 + "\n\n")
print(f" Hoàn tất toàn bộ quy trình. Báo cáo kỹ thuật được lưu tại: {report_path}")


     KẾT QUẢ ĐÁNH GIÁ
Macro F1 - Tập Huấn luyện (Train) : 0.7744
Macro F1 - Tập Kiểm định  (Val)   : 0.7556
Macro F1 - Tập Kiểm thử   (Test)  : 0.7597
Khoảng sai lệch tổng quát hóa     : 0.0188

>> [5/5] Lưu báo cáo chi tiết...
 Hoàn tất toàn bộ quy trình. Báo cáo kỹ thuật được lưu tại: /content/drive/MyDrive/CS114/TeamModel/reports/lr_lsa_TFIDF_report.txt


In [15]:
splits = [
        ("TRAIN SET", y_train, preds_train_final),
        ("VALIDATION SET", y_val, preds_val_final),
        ("TEST SET", y_test, preds_test_final),
    ]

for name, y_true, y_pred in splits:
        section = f"--- ĐÁNH GIÁ {name} ---\n"
        section += classification_report(y_true, y_pred, digits=4)
        section += "\n" + "=" * 80 + "\n\n"
        print(section)

--- ĐÁNH GIÁ TRAIN SET ---
              precision    recall  f1-score   support

     anxiety     0.8178    0.7689    0.7926      3899
  depression     0.7120    0.7002    0.7060     10153
      normal     0.8918    0.9205    0.9059     12693
    suicidal     0.6933    0.6927    0.6930      7840

    accuracy                         0.7871     34585
   macro avg     0.7787    0.7706    0.7744     34585
weighted avg     0.7857    0.7871    0.7862     34585



--- ĐÁNH GIÁ VALIDATION SET ---
              precision    recall  f1-score   support

     anxiety     0.8073    0.7594    0.7826       557
  depression     0.6866    0.6809    0.6837      1451
      normal     0.8777    0.9212    0.8989      1815
    suicidal     0.6707    0.6438    0.6569      1120

    accuracy                         0.7696      4943
   macro avg     0.7606    0.7513    0.7556      4943
weighted avg     0.7668    0.7696    0.7678      4943



--- ĐÁNH GIÁ TEST SET ---
              precision    recall  f1-sco